# W2D4: Train/Test Split & Cross-Validation

## Objective

The objective of this task is to understand and implement:

- Train/Test Split
- Feature Scaling
- StandardScaler
- MinMaxScaler
- RobustScaler
- Cross-Validation
- Stratified K-Fold Cross-Validation

A Logistic Regression model will be used to evaluate the effect of different scaling techniques.

In [1]:
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, StratifiedKFold

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1. Load Dataset

The Breast Cancer Wisconsin dataset available in Scikit-learn is used for this task.

It is a binary classification dataset containing numerical features.

In [2]:
data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset loaded successfully")
print("Number of samples:", X.shape[0])
print("Number of features:", X.shape[1])

Dataset loaded successfully
Number of samples: 569
Number of features: 30


## 2. Train/Test Split

The dataset is divided into training and testing sets.

- 80% of the data is used for training.
- 20% of the data is used for testing.
- `random_state=42` ensures reproducible results.
- Stratification maintains a similar class distribution in both sets.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train/Test Split completed")
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Train/Test Split completed
Training samples: 455
Testing samples: 114


## 3. Feature Scaling

Feature scaling puts numerical features on comparable scales.

Three scaling techniques are compared:

1. StandardScaler
2. MinMaxScaler
3. RobustScaler

In [4]:
scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler()
}

print("Three scalers created successfully:")
for name in scalers:
    print("-", name)

Three scalers created successfully:
- StandardScaler
- MinMaxScaler
- RobustScaler


## 4. Compare Feature Scaling Methods

Three scaling methods are applied with Logistic Regression:

- **StandardScaler**: Standardizes features using mean and standard deviation.
- **MinMaxScaler**: Scales features to a range between 0 and 1.
- **RobustScaler**: Uses median and interquartile range and is less sensitive to outliers.

A Pipeline is used to prevent data leakage during preprocessing.

In [5]:
results = {}

for scaler_name, scaler in scalers.items():

    pipeline = Pipeline([
        ("scaler", scaler),
        ("model", LogisticRegression(max_iter=5000))
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    results[scaler_name] = accuracy

    print(f"{scaler_name}: {accuracy:.4f}")

StandardScaler: 0.9825
MinMaxScaler: 0.9561
RobustScaler: 0.9825


## 5. 5-Fold Stratified Cross-Validation

Cross-validation is used to obtain a more reliable estimate of model performance.

Stratified K-Fold is used because this is a classification problem. Five folds are created, and each fold is used as validation data once.

In [6]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = {}

for scaler_name, scaler in scalers.items():

    pipeline = Pipeline([
        ("scaler", scaler),
        ("model", LogisticRegression(max_iter=5000))
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    cv_results[scaler_name] = scores.mean()

    print(f"\n{scaler_name}")
    print("Fold scores:", np.round(scores, 4))
    print("Mean accuracy:", round(scores.mean(), 4))
    print("Standard deviation:", round(scores.std(), 4))


StandardScaler
Fold scores: [0.9737 0.9474 0.9649 0.9912 0.9912]
Mean accuracy: 0.9737
Standard deviation: 0.0166

MinMaxScaler
Fold scores: [0.9825 0.9298 0.9561 0.9825 0.9735]
Mean accuracy: 0.9649
Standard deviation: 0.02

RobustScaler
Fold scores: [0.9825 0.9561 0.9737 0.9912 0.9912]
Mean accuracy: 0.9789
Standard deviation: 0.0131


## 6. Final Model

StandardScaler with Logistic Regression is used as the final model.

The model is trained on the training data and evaluated on the unseen test data.

In [7]:
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

final_model.fit(X_train, y_train)

final_predictions = final_model.predict(X_test)

final_accuracy = accuracy_score(
    y_test,
    final_predictions
)

print("Final Model Accuracy:", round(final_accuracy, 4))

Final Model Accuracy: 0.9825


## 7. Results Summary

The three feature scaling techniques were evaluated using Logistic Regression.

The test-set accuracy and 5-fold cross-validation results were compared.

The scaling method with the highest cross-validation mean accuracy provides the best performance for this experiment.

## 8. Conclusion

In this task, Train/Test Split and Cross-Validation were successfully implemented.

StandardScaler, MinMaxScaler, and RobustScaler were compared using Logistic Regression.

5-fold Stratified Cross-Validation was used to obtain a reliable estimate of model performance.

A Pipeline was used to prevent data leakage during scaling and cross-validation.

Feature scaling is especially important for distance-based and many gradient-based machine learning algorithms because features with larger numerical ranges can otherwise have a greater influence on the model.